In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# กำหนดค่า noise management ด้วย Estimator

<Accordion>
<AccordionItem title="เวอร์ชันแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
มีหลายวิธีในการจัดการ noise โดยทั่วไปโดยใช้เทคนิค error mitigation และ error suppression ต่างๆ เพื่อหลีกเลี่ยงข้อผิดพลาดก่อนที่จะเกิดขึ้น เทคนิคเหล่านี้มักทำให้เกิดค่าใช้จ่ายในการประมวลผลล่วงหน้า ดังนั้น จึงสำคัญที่จะต้องหาจุดสมดุลระหว่างการปรับปรุงผลลัพธ์และการทำให้งานเสร็จในเวลาที่เหมาะสม

Estimator รองรับเทคนิค noise management ต่อไปนี้ ดู [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques) สำหรับคำอธิบายของแต่ละเทคนิค ดูส่วน [Custom error settings](#advanced-error) สำหรับคำแนะนำการเปิดใช้งานเทคนิคเหล่านี้

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Resilience level

`resilience_level` ระบุระดับความทนทานต่อ errors ที่ต้องการสร้าง ระดับที่สูงกว่าให้ผลลัพธ์ที่แม่นยำกว่า โดยแลกกับเวลาประมวลผลที่นานกว่า Resilience levels สามารถใช้กำหนดค่าการแลกเปลี่ยนระหว่างต้นทุน/ความแม่นยำเมื่อใช้ noise management กับ primitive query Noise management ลด errors (bias) ในผลลัพธ์โดยประมวลผล outputs จากกลุ่มหรือ ensemble ของ circuits ที่เกี่ยวข้อง ระดับการลด errors ขึ้นอยู่กับวิธีที่ใช้ resilience level abstract การเลือกวิธี noise management โดยละเอียดเพื่อให้ผู้ใช้สามารถพิจารณาการแลกเปลี่ยนต้นทุน/ความแม่นยำที่เหมาะสมกับ application ของตน

ด้วยเหตุนี้ แต่ละระดับจึงสอดคล้องกับวิธีหรือหลายวิธีที่มีระดับ quantum sampling overhead เพิ่มขึ้น เพื่อให้คุณทดสอบการแลกเปลี่ยนระหว่างเวลา/ความแม่นยำที่แตกต่างกัน ตารางต่อไปนี้แสดงระดับและวิธีที่สอดคล้องกันซึ่งมีสำหรับแต่ละ primitive

<span id="resilience-table"></span>

| Resilience Level | คำอธิบาย | เทคนิค |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | ไม่มี mitigation | ไม่มี |
| 1 [ค่าเริ่มต้น] | ต้นทุน mitigation ขั้นต่ำ: ลด errors ที่เกี่ยวข้องกับ readout errors | [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) measurement twirling |
| 2 | ต้นทุน mitigation ปานกลาง โดยทั่วไปลด bias ใน estimators แต่ไม่รับประกันว่าจะ zero-bias | ระดับ 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) และ gate twirling

> **Info:** Resilience levels ขณะนี้อยู่ในระยะ beta ดังนั้น sampling overhead และคุณภาพของโซลูชันจะแตกต่างกันไปตาม circuit คุณสมบัติใหม่ ตัวเลือกขั้นสูง และเครื่องมือการจัดการจะถูกเผยแพร่แบบต่อเนื่อง ไม่รับประกันว่าวิธี noise management เฉพาะจะถูกใช้ที่แต่ละ resilience level

### กำหนดค่า Estimator ด้วย resilience levels
คุณสามารถใช้ resilience levels เพื่อระบุเทคนิค noise management หรือตั้งค่าเทคนิคที่กำหนดเองทีละรายการตามที่อธิบายใน [Custom error settings](#advanced-error)

> **Note:** ตัวเลือกใดๆ ที่คุณระบุด้วยตนเองนอกเหนือจาก resilience level จะถูกใช้เพิ่มเติมจากชุดตัวเลือกพื้นฐานที่กำหนดโดย resilience level ดังนั้น ในหลักการ คุณสามารถตั้งค่า resilience level เป็น 1 แต่ปิด measurement mitigation แม้ว่าจะไม่แนะนำ
> 
> ตัวอย่างเช่น การตั้งค่า resilience level เป็น 0 จะปิด `zne_mitigation` แต่ `estimator.options.resilience.zne_mitigation = True` จะ override ค่านั้น

### ตัวอย่าง
โค้ดต่อไปนี้เปิดใช้งาน ZNE, TREX และ gate twirling โดยการตั้งค่า `resilience_level 2`

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## การตั้งค่า noise management แบบกำหนดเอง
คุณสามารถเปิด/ปิดวิธี noise management แต่ละรายการโดยใช้ [Estimator options](/guides/estimator-options)

> **Note:** ตัวเลือกทั้งหมดไม่ได้ทำงานร่วมกันบน circuit ทุกประเภท ดูรายละเอียดที่ [feature compatibility table](estimator-options#options-compatibility-table)

### ตัวอย่าง

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>